# Data Validation & Quality Assessment for Scraped News
## LimeoX assignment: results notebook

This notebook documents and demonstrates the validation stage built for the article dataset (100,026 articles, PostgreSQL table `public.articles_cleaned`).

**How the work is organised**

- The rules are real Python modules in this folder (`config.py`, `validation_rules.py`, `report_generator.py`, `validate.py`), covered by 59 `pytest` tests.
- The full run over all 100,026 articles is done by `python validate.py --run`, which writes the reports into `validation_output/`.
- **This notebook** explains each rule, demonstrates the rules and edge cases on sample records, reads the real reports, and answers Exercises A to E from the lecture.

**To run it:** put the notebook in the project folder (next to `validate.py`), start it from there, and run the cells top to bottom. It needs `pandas` (`python -m pip install pandas notebook`). It never connects to the database and never changes data.

> Validation does not mean every record must be perfect. Each record is assessed against documented rules, with transparent decisions about what can be accepted, reviewed, or quarantined.

## 1. Preprocessing versus validation

| | Preprocessing (previous task) | Validation (this task) |
|---|---|---|
| Purpose | Transform the data | Check the data against a contract |
| Example | Repair `PokÃ©mon` to `Pokémon`, tidy whitespace | Is the title non-empty and long enough? Is the URL valid? |
| Changes data? | Yes, writes `articles_cleaned` | **No.** Read-only; it only labels rows and writes reports |
| Result | 100,026 cleaned rows plus quality flags | ACCEPT, WARNING or QUARANTINE per row, plus an issue log |

Validation is repeatable: the same input and the same configuration give the same results. The validation stage reuses the preprocessing flags (`is_short`, `is_non_article`, `has_html_tags`, `has_boilerplate`, `has_suspicious_encoding`, `is_duplicate`) instead of recomputing them.

## 2. Setup

In [ ]:
import json
import sys
from datetime import date, datetime, time, timedelta, timezone
from pathlib import Path

import pandas as pd

BASE = Path.cwd()                      # start Jupyter from the project folder
sys.path.insert(0, str(BASE))

import config                          # the data contract and thresholds
import validation_rules as vr          # the rules themselves

OUT = BASE / "validation_output"
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 80)

print("Project folder:", BASE)
print("Report folder :", OUT, "(found)" if OUT.exists() else "(NOT FOUND: run  python validate.py --run  first)")

## 3. Data contract

A **data contract** documents the expected schema, types and constraints. It lives in `config.py` so that no rule contains magic numbers.

Mapping from the lecture's example fields to the real database columns:

| Lecture field | Real column | Note |
|---|---|---|
| `news_url_clean` | `url` | Unique in the database, so only *normalised* repeats can occur |
| `title_clean` | `title` | |
| `content_clean` | `body` | |
| `author_clean` | `author` | Optional; missing is INFO only |
| `published_at_utc` | `published_date` (DATE) and `published_time` (TIMETZ) | Date is always present; time is partial |
| `domain` | `source` | Stored without `www.`, compared with the URL hostname |
| `scraped_at_utc` | `imported_at` | This is the **load time** into the database, not the true scrape time |

In [ ]:
print("Ruleset version :", config.RULESET_VERSION)
print("Input table     :", config.INPUT_TABLE, "(read-only)")

print("\nREQUIRED columns (missing column = schema failure, the run stops):")
for name, types in config.REQUIRED_COLUMNS.items():
    print(f"  {name:6} accepted types: {sorted(types)}")

print("\nOPTIONAL columns (missing column = INFO, dependent rules are skipped):")
print(" ", ", ".join(config.OPTIONAL_COLUMNS))

print("\nThresholds:")
print("  title shorter than        :", config.MIN_TITLE_CHARS_WARNING, "characters -> WARNING")
print("  body shorter than         :", config.MIN_CONTENT_CHARS_WARNING, "characters -> WARNING")
print("  earliest plausible date   :", config.EARLIEST_PLAUSIBLE_DATE)
print("  block-page text checked in bodies up to:", config.ERROR_PATTERN_MAX_CHARS, "characters")
print("\nFATAL rules (-> QUARANTINE):", sorted(config.FATAL_RULES))
print("Every other issue            -> WARNING; no issue -> ACCEPT")

## 4. The rules, explained

Each issue has a stable rule ID, a severity (`ERROR`, `WARNING`, `INFO`), the affected article id and field, and a message.

| Rule | Severity | What it checks and why |
|---|---|---|
| SCHEMA_001 | ERROR | Duplicate column names. Stops the run, because the table layout is ambiguous. |
| SCHEMA_002 | ERROR | A required column is missing or has the wrong type. Reported, never silently created. |
| SCHEMA_003 | INFO | An optional column is missing. Rules that depend on it are skipped and listed in the summary. |
| SCHEMA_004 | WARNING/INFO | An optional column exists but has an unexpected type (added for this database). |
| FIELD_001 | ERROR (fatal) | `url`, `title` or `body` is missing, blank, or only invisible characters. The article cannot be used. |
| FIELD_002 | INFO | Author is missing. Never guessed and never fatal: many real sources print no byline. |
| TYPE_001 | WARNING | A text field holds a non-text value. |
| URL_001 | ERROR (fatal) | Not a plausible http(s) URL (bad scheme, no hostname, spaces, bad port). Syntax only; the page is not contacted. |
| URL_002 | WARNING | `source` does not match the URL hostname. `www.`, case and subdomains are ignored. |
| TIME_001 | INFO | `imported_at` is missing, so traceability is reduced. |
| TIME_002 | WARNING | A timestamp is present but cannot be parsed. |
| TIME_003 | WARNING | Published later than imported. Exact comparison when the time has a UTC offset; otherwise date-only with a one-day tolerance. |
| TIME_004 | WARNING | Publication date before 1990 (implausible). |
| TIME_005 | INFO | Publication date is missing. |
| TEXT_001 | WARNING | Title shorter than 8 characters. |
| TEXT_002 | WARNING | Body shorter than 200 characters. A warning, not an error: short breaking-news items can be legitimate. |
| TEXT_003 | WARNING | Body looks like a block page, CAPTCHA or access-denied text. Only short bodies are checked, because a long article may merely mention these words. |
| TEXT_004 | WARNING | Placeholder title such as "Home" or "Untitled", or a title with no letters or digits (Exercise A). |
| TEXT_005 | WARNING | Body is mostly punctuation or symbols. |
| FLAG_001 to FLAG_005 | WARNING | Reused preprocessing flags: short text (under 50 words), non-article page, HTML-like tags, cookie or newsletter text, broken encoding. |
| DUP_001 | WARNING | The normalised URL (no `www.`, no tracking parameters, sorted query) occurs more than once. |
| DUP_002 | WARNING | Identical title and body: strong duplicate candidate. |
| DUP_003 | WARNING | Identical body with a different title: duplicate candidate. |

Duplicates are only **candidates for review**. Nothing is merged or deleted, because a repeated URL can be an updated version of the same article.

## 5. The rules on sample records (Exercises A and D)

Each demo record is built to trigger one specific behaviour. The code below runs the real rule functions, `validate_row` and `disposition`, and then **asserts** the expected outcome. If a rule were changed in a way that breaks an expectation, this cell would fail.

In [ ]:
AVAILABLE = {"id", "url", "title", "body", "source", "author",
             "published_date", "published_time", "imported_at"}
LONG_BODY = "Markets rallied after several firms reported earnings. " * 8      # about 440 characters
IMPORTED = datetime(2026, 10, 27, 10, 0, tzinfo=timezone.utc)

def make(id, **changes):
    row = dict(id=id, url="https://news.example.com/story/%d" % id,
               title="Markets rally after earnings", body=LONG_BODY,
               source="news.example.com", author="A. Writer",
               published_date=date(2026, 10, 27), published_time=time(9, 30, tzinfo=timezone.utc),
               imported_at=IMPORTED)
    row.update(changes)
    return row

# (record, what it demonstrates, expected rule codes, expected disposition)
CASES = [
    (make(1),                                         "clean record",                              set(),                       "ACCEPT"),
    (make(2, url="not-a-url"),                        "invalid URL",                               {"URL_001"},                 "QUARANTINE"),
    (make(3, title=None),                             "missing title",                             {"FIELD_001"},               "QUARANTINE"),
    (make(4, title="   ", body=""),                   "blank title and body",                      {"FIELD_001"},               "QUARANTINE"),
    (make(5, title="Home"),                           "placeholder title (Exercise A)",            {"TEXT_001", "TEXT_004"},    "WARNING"),
    (make(6, title="!!! ???"),                        "punctuation-only title (Exercise A)",       {"TEXT_001", "TEXT_004"},    "WARNING"),
    (make(7, body="Access denied. Enable JavaScript."), "block page text",                         {"TEXT_002", "TEXT_003"},    "WARNING"),
    (make(8, url="https://news.example.com:8443/story?id=8&utm_source=x#top"), "port, query string and fragment are valid", set(), "ACCEPT"),
    (make(9, title="\u00dcberraschung: B\u00f6rsen steigen nach Gewinnen", author=""), "Unicode title, empty author", {"FIELD_002"}, "ACCEPT"),
    (make(10, published_date=date(2026, 10, 30)),     "date-only publication 3 days after import", {"TIME_003"},               "WARNING"),
    (make(11, published_date=date(2026, 10, 27),
          published_time=time(16, 0, tzinfo=timezone(timedelta(hours=5)))),
                                                      "16:00 at UTC+5 = 11:00 UTC, after 10:00 import", {"TIME_003"},           "WARNING"),
    (make(12, published_date=date(1985, 5, 1)),       "implausibly old date",                      {"TIME_004"},                "WARNING"),
    (make(13, source="other-site.com"),               "source does not match URL host",            {"URL_002"},                 "WARNING"),
    (make(14, published_date=None, imported_at=None), "missing dates are INFO, never a warning",    {"TIME_005", "TIME_001"},    "ACCEPT"),
]

table = []
for row, why, expected_codes, expected_disp in CASES:
    issues_found = vr.validate_row(row, AVAILABLE)
    codes = {i["rule_id"] for i in issues_found}
    disp = vr.disposition(issues_found)
    assert codes == expected_codes, (row["id"], why, codes, expected_codes)
    assert disp == expected_disp, (row["id"], why, disp, expected_disp)
    table.append({"id": row["id"], "demonstrates": why, "rule codes": ";".join(sorted(codes)) or "-", "disposition": disp})

display(pd.DataFrame(table))
print("All", len(CASES), "sample records behave as expected.")

### Edge cases of the helper functions (Exercise D)

URLs with ports and query strings, Unicode, empty fields, timestamps with timezone offsets, and publication later than import time.

In [ ]:
# --- URL syntax ---------------------------------------------------------------------
assert vr.is_valid_http_url("https://example.com/story/1")
assert vr.is_valid_http_url("http://news.example.org/")
assert vr.is_valid_http_url("https://example.com:8080/a?x=1#section")      # port, query, fragment
assert vr.is_valid_http_url("https://b\u00fccher.example/stra\u00dfe")        # Unicode host and path
assert not vr.is_valid_http_url("not-a-url")
assert not vr.is_valid_http_url("ftp://example.com/file")                    # wrong scheme
assert not vr.is_valid_http_url("https://example..com/x")                    # empty label
assert not vr.is_valid_http_url("https://exa mple.com/x")                    # space
assert not vr.is_valid_http_url(None)

# --- empty / missing values ---------------------------------------------------------
assert vr.is_missing(None) and vr.is_missing("") and vr.is_missing("   ")
assert vr.is_missing("\u200b\u200b")                                          # only invisible characters
assert not vr.is_missing("Article title")

# --- same address, different spelling (duplicate detection key) ---------------------
a = vr.normalize_url("https://www.Example.com/a/?utm_source=x&b=2&a=1#top")
b = vr.normalize_url("http://example.com/a?a=1&b=2")
assert a == b, (a, b)

# --- placeholder titles -------------------------------------------------------------
for t in ["Home", "Untitled", "Page not found", "!!!", "--"]:
    assert vr.is_placeholder_title(t), t
for t in ["Markets rally after earnings", "Home prices fall again"]:
    assert not vr.is_placeholder_title(t), t

# --- source versus hostname ---------------------------------------------------------
assert vr.domain_matches("bbc.com", "www.bbc.com")
assert vr.domain_matches("bbc.com", "news.bbc.com")                          # subdomain is fine
assert not vr.domain_matches("bbc.com", "cnn.com")

print("All edge-case assertions passed.")

## 6. Results of the real run

The cells below only **read** the reports written by `python validate.py --run`. Nothing is recalculated and the database is not touched.

In [ ]:
need = ["data_validation_all.csv", "data_validation_issues.csv", "data_validation_summary.json",
        "data_field_completeness.csv", "domain_quality_summary.csv"]
absent = [f for f in need if not (OUT / f).exists()]
assert not absent, "Run 'python validate.py --run' first. Missing: %s" % absent

summary = json.loads((OUT / "data_validation_summary.json").read_text(encoding="utf-8"))
all_rows = pd.read_csv(OUT / "data_validation_all.csv", low_memory=False)
issues = pd.read_csv(OUT / "data_validation_issues.csv", low_memory=False, dtype={"group": str})
issues["article_id"] = pd.to_numeric(issues["article_id"], errors="coerce")

print("Ruleset version     :", summary["run"]["ruleset_version"])
print("Rows in input table :", f"{summary['input_rows_in_table']:,}")
print("Rows validated      :", f"{summary['input_rows_validated']:,}")
print("Issues logged       :", f"{summary['issues']['total']:,}")

### 6.1 Dispositions: ACCEPT, WARNING, QUARANTINE

In [ ]:
disp = (all_rows["validation_disposition"].value_counts()
        .reindex(["ACCEPT", "WARNING", "QUARANTINE"], fill_value=0))
result = pd.DataFrame({"articles": disp, "share_pct": (disp * 100 / len(all_rows)).round(2)})
display(result)
assert disp.sum() == len(all_rows)
print("The three partitions add up to", f"{disp.sum():,}", "= all validated rows (no overlap, no gaps).")

Note that one article can trigger several rules, so the **issue count is not the same as the number of affected articles**. The table below shows both.

In [ ]:
by_rule = (issues[issues["article_id"].notna()]
           .groupby(["rule_id", "severity"])
           .agg(issues=("rule_id", "size"), distinct_articles=("article_id", "nunique"))
           .reset_index().sort_values("distinct_articles", ascending=False))
display(by_rule)

### 6.2 Field completeness

In [ ]:
display(pd.read_csv(OUT / "data_field_completeness.csv"))

### 6.3 Exercise B: domain-level monitoring

For each source domain: article count, invalid URL rate, missing publication date rate, and short-content rate. Sources with a high rate point to a scraper problem, not to bad luck.

In [ ]:
domains = pd.read_csv(OUT / "domain_quality_summary.csv")
cols = [c for c in ["source", "articles", "warning_pct", "invalid_url_rate_pct",
                    "missing_published_date_rate_pct", "short_content_rate_pct"] if c in domains.columns]

print("Largest sources:")
display(domains.sort_values("articles", ascending=False)[cols].head(10))

if "short_content_rate_pct" in domains.columns:
    print("\nHighest short-content rate (sources with at least 100 articles):")
    display(domains[domains["articles"] >= 100].sort_values("short_content_rate_pct", ascending=False)[cols].head(10))

## 7. Exercise C: duplicate policy

**Task:** for repeated candidates, select the record with the **longest non-empty content**, keep an **audit log** of the discarded candidates, and explain how this might mishandle article updates.

The cell below applies that policy to every duplicate group found by DUP_001, DUP_002 and DUP_003. It is a *simulation*: it only writes an audit log. It does not delete anything. Ties are broken by the lowest article id.

In [ ]:
dup_rules = ["DUP_001", "DUP_002", "DUP_003"]
dups = issues[issues["rule_id"].isin(dup_rules) & issues["group"].notna() & issues["article_id"].notna()].copy()
dups["article_id"] = dups["article_id"].astype(int)

words = all_rows.set_index("article_id")["clean_word_count"]
dups["words"] = dups["article_id"].map(words).fillna(0).astype(int)

audit_rows = []
groups_total, groups_kept_not_first = {}, {}
for (rule, group), part in dups.groupby(["rule_id", "group"]):
    ordered = part.sort_values(["words", "article_id"], ascending=[False, True])
    keep = ordered.iloc[0]
    groups_total[rule] = groups_total.get(rule, 0) + 1
    if keep["article_id"] != part["article_id"].min():
        groups_kept_not_first[rule] = groups_kept_not_first.get(rule, 0) + 1
    for _, gone in ordered.iloc[1:].iterrows():
        audit_rows.append({"rule_id": rule, "group": group,
                           "kept_article_id": int(keep["article_id"]), "kept_words": int(keep["words"]),
                           "discarded_article_id": int(gone["article_id"]), "discarded_words": int(gone["words"])})

audit = pd.DataFrame(audit_rows, columns=["rule_id", "group", "kept_article_id", "kept_words",
                                          "discarded_article_id", "discarded_words"])
audit.to_csv(OUT / "data_duplicate_audit_log.csv", index=False, encoding="utf-8")

overview = pd.DataFrame({
    "duplicate_groups": pd.Series(groups_total),
    "discarded_candidates": audit.groupby("rule_id").size() if len(audit) else pd.Series(dtype=int),
    "groups_where_longest_is_not_the_lowest_id": pd.Series(groups_kept_not_first),
}).fillna(0).astype(int)
display(overview)
print("Audit log written to:", OUT / "data_duplicate_audit_log.csv", f"({len(audit):,} discarded candidates)")
display(audit.head(10))

### How this policy can mishandle article updates

"Longest content wins" assumes that copies are exact duplicates and that more text is better. For **updated articles** this is risky:

1. **Length is not recency.** An updated version is often *shorter* than the first one (a correction, a trimmed intro, a removed paragraph), so the policy keeps the outdated version and discards the corrected one.
2. **Live blogs and developing stories** grow over time. If both copies were scraped at different moments, the policy may keep a copy that is only a snapshot, and the discarded copy still carries a different, valid state of the story.
3. **Truncated scrapes.** A copy cut short by a paywall or a failed extraction loses to the full copy, which is right; but a copy with extra boilerplate (cookie text, related-links lists) wins on length without being a better article.
4. **A repeated URL can be a different article.** Sites reuse URLs for recurring pages (daily summaries, live pages). The discarded record would then be lost information, not a duplicate.
5. **Ties and near-equal lengths** make the choice arbitrary.

**Mitigations used in this project:** nothing is deleted from the database; every discarded candidate is written to the audit log with its length; the ML-ready export keeps the *first* copy (lowest id) and lists every exclusion in `data_ml_excluded.csv`; and a human can review the candidates because duplicates stay in the issue log as DUP rules. A safer production policy would also compare `imported_at` and text similarity before discarding a copy, and would keep both versions when the texts differ.

## 8. Exercise E: acceptance criteria for two tasks

What is *fatal*, what is only a *warning*, and what is *optional* depends on what the data is used for.

| | **Full-text classification** | **Title-only click prediction** |
|---|---|---|
| **Fatal** (QUARANTINE) | `url`, `title` or `body` missing or blank (FIELD_001); invalid URL (URL_001) | `title` missing or blank (FIELD_001 on title); placeholder or punctuation-only title (TEXT_004) |
| **Warning** (review) | short body, block-page text, mostly punctuation, non-article page, leftover HTML, cookie text, broken encoding, short title, duplicates, source/URL mismatch, implausible dates | short title (TEXT_001), non-article page, broken encoding, duplicates, source/URL mismatch, implausible dates |
| **Optional** (ignored) | author, publication date and time, `imported_at`, description, tags | `body` and every body check (short, block page, punctuation, HTML, cookie text), URL validity, author, dates |
| **Reasoning** | The model learns from the text, so the text must exist and be real. | The model only sees the headline; a missing body does not matter, but a fake headline such as "Home" would poison the training data. |

The cell below applies both policies to the real issue log and shows how many articles each one would accept, send to review, or quarantine. The pipeline itself uses the full-text policy.

In [ ]:
POLICIES = {
    "Full-text classification": {
        "fatal": {("FIELD_001", "url"), ("FIELD_001", "title"), ("FIELD_001", "body"), ("URL_001", "url")},
        "ignored_rules": {"FIELD_002", "TIME_001", "TIME_005"},
        "ignored_pairs": set(),
    },
    "Title-only click prediction": {
        "fatal": {("FIELD_001", "title"), ("TEXT_004", "title")},
        "ignored_rules": {"FIELD_002", "TIME_001", "TIME_005", "URL_001", "TEXT_002", "TEXT_003", "TEXT_005",
                          "FLAG_001", "FLAG_003", "FLAG_004"},
        "ignored_pairs": {("FIELD_001", "body"), ("FIELD_001", "url")},
    },
}

row_issues = issues[issues["article_id"].notna() & (issues["severity"] != "INFO")].copy()
total = len(all_rows)
rows = []
for name, policy in POLICIES.items():
    pairs = list(zip(row_issues["rule_id"], row_issues["field"]))
    is_fatal = pd.Series([p in policy["fatal"] for p in pairs], index=row_issues.index)
    is_ignored = pd.Series([(p[0] in policy["ignored_rules"]) or (p in policy["ignored_pairs"]) for p in pairs],
                           index=row_issues.index)
    quarantine_ids = set(row_issues.loc[is_fatal, "article_id"])
    warning_ids = set(row_issues.loc[~is_fatal & ~is_ignored, "article_id"]) - quarantine_ids
    rows.append({"policy": name, "ACCEPT": total - len(quarantine_ids) - len(warning_ids),
                 "WARNING": len(warning_ids), "QUARANTINE": len(quarantine_ids)})
display(pd.DataFrame(rows).set_index("policy"))

## 9. Raw, cleaned and validated data

Validation labels data; it does not remove it. The reconciliation shows that no article is lost between the stages, and that the only reduction happens in the separate ML-ready export.

In [ ]:
def show_json(path, limit=14):
    data = json.loads(Path(path).read_text(encoding="utf-8"))
    print("==", Path(path).name)
    for key, value in data.items():
        if isinstance(value, dict):
            print(f"  {key}:")
            items = sorted(value.items(), key=lambda kv: -kv[1] if isinstance(kv[1], (int, float)) else 0)
            for k, v in items[:limit]:
                print(f"      {k}: {v}" if not isinstance(v, (dict, list)) else f"      {k}: ...")
        elif isinstance(value, list):
            print(f"  {key}: {len(value)} items")
        else:
            print(f"  {key}: {value}")

for name in ["data_validation_reconciliation.json", "data_ml_ready_summary.json"]:
    if (OUT / name).exists():
        show_json(OUT / name)
    else:
        print("==", name, "not found (run: python validate.py --verify / --export-ml-ready)")

Result of the full run (see the cells above for the exact numbers on your machine):

| Stage | Articles | What changed |
|---|---:|---|
| Raw (`public.articles`) | 100,026 | Untouched source data |
| Cleaned (`public.articles_cleaned`) | 100,026 | Text repaired and tidied; flags added |
| Validated | 100,026 | **No data changed.** 93,311 ACCEPT, 6,715 WARNING, 0 QUARANTINE |
| ML-ready export (`data_ml_ready.csv`) | 95,094 | 4,932 articles excluded, each with its reasons in `data_ml_excluded.csv` |

## 10. Key takeaways

- Preprocessing transforms data; validation checks it against a contract and leaves it unchanged.
- Rules are explicit, configurable (`config.py`), and tested (59 `pytest` tests).
- Only a missing or blank required field (FIELD_001) and an invalid URL (URL_001) quarantine an article. A missing author or publication date is never fatal.
- A syntactically valid URL does not prove that a page is accessible; no website is contacted.
- Duplicate URLs and texts are reported as candidates, because a repeat can be an updated article.
- Issue logs, warning rows and the duplicate audit log are kept for auditability.
- Quality is monitored per source domain, which exposed sources that return pages that are not articles.

### Deliberate differences from the lecture's example code

- **No pandas in the pipeline.** 100,026 rows are streamed in batches of 2,000 with the standard library; pandas is used only in this notebook to display the reports.
- **URL_002 ignores `www.`, case and subdomains**, because the stored `source` has no `www.`; an exact string comparison would raise false warnings for many rows.
- **Block-page text (TEXT_003) is searched only in bodies up to 1,500 characters**, so a long article that merely mentions "captcha" is not flagged.
- **Additional rules and outputs** (SCHEMA_004, TIME_004/005, TEXT_004/005, DUP_003, FLAG_001 to FLAG_005, the ML-ready export and `--verify`) were added for this dataset.